# Crawl CareerViet: hiện tại và lịch sử

Chạy lần lượt từ trên xuống. Chọn `MODE` ở phần cấu hình: `live`, `history`, hoặc `both` (mặc định). Mỗi nguồn xử lý tối đa 50 URL chưa xử lý trong một lượt. Dùng `None` để crawl hết.

Crawler dùng `curl_cffi`, một luồng và có checkpoint. Dữ liệu hiện tại lấy từ sitemap CareerViet; dữ liệu lịch sử lấy HTML snapshot Wayback của cả CareerBuilder và CareerViet. Hai nguồn lưu file riêng.


## 1. Môi trường

Chọn Python kernel của dự án. Nếu thiếu thư viện, bỏ dấu `#` và chạy cell cài đặt một lần. Sau khi cập nhật code crawler, restart kernel trước khi chạy lại notebook.


In [ ]:
# %pip install pandas beautifulsoup4 curl_cffi tqdm


In [6]:
import sys
from pathlib import Path

# Tự tìm thư mục dự án khi mở notebook từ root hoặc notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/crawler/careerviet_crawler.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Hãy mở notebook trong thư mục project_intro_ds hoặc notebooks của dự án.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.crawler import CareerVietCrawler

print("Project:", PROJECT_ROOT)


Project: g:\hoc_tap\IntroDS\project_intro_ds


## 2. Cấu hình chung

`HISTORY_START_YEAR` và `HISTORY_END_YEAR` lọc **năm chụp Wayback**, không phải năm đăng tin. Mặc định 2022–2025. Danh sách lịch sử đã lưu được bổ sung khi mở rộng khoảng năm; muốn bộ dữ liệu riêng, đổi đường dẫn file lịch sử.

Giữ `RESUME_JOBS = True` để chạy tiếp. `False` ghi đè CSV kết quả của nguồn được chọn. `REFRESH_LIVE_URLS` chỉ gom lại sitemap hiện tại. Gom lịch sử luôn chạy tiếp theo checkpoint CDX.


In [8]:
MODE = "both"                 # "live", "history", "both"
CRAWL_LIMIT = None              # Giới hạn URL mỗi nguồn, mỗi lượt; None = tất cả
RESUME_JOBS = True
SAVE_EVERY = 50
REFRESH_LIVE_URLS = False

HISTORY_START_YEAR = 2022
HISTORY_END_YEAR = 2025
LIVE_REQUEST = dict(min_delay=2.5, max_delay=5.0, timeout=30)
HISTORY_REQUEST = dict(min_delay=5.0, max_delay=10.0, timeout=60)

RAW_DIR = PROJECT_ROOT / "data/raw"
URL_FILE = RAW_DIR / "careerviet_it_job_urls.csv"
JOB_FILE = RAW_DIR / "careerviet_it_jobs.csv"
HISTORY_URL_FILE = RAW_DIR / "careerviet_history_urls.csv"
HISTORY_JOB_FILE = RAW_DIR / "careerviet_history_jobs.csv"

if MODE not in {"live", "history", "both"}:
    raise ValueError("MODE phải là live, history hoặc both")
RUN_LIVE = MODE in {"live", "both"}
RUN_HISTORY = MODE in {"history", "both"}
print(f"Chế độ: {MODE}; giới hạn mỗi nguồn: {CRAWL_LIMIT}")
print(f"Khoảng năm chụp lịch sử: {HISTORY_START_YEAR}–{HISTORY_END_YEAR}")
print("Thư mục kết quả:", RAW_DIR)


Chế độ: both; giới hạn mỗi nguồn: None
Khoảng năm chụp lịch sử: 2022–2025
Thư mục kết quả: g:\hoc_tap\IntroDS\project_intro_ds\data\raw


## 3. Gom URL hiện tại

Cell tự bỏ qua nếu chọn `history`. Dùng lại danh sách URL hiện có, trừ khi bật `REFRESH_LIVE_URLS`.


In [9]:
if RUN_LIVE:
    if URL_FILE.exists() and not REFRESH_LIVE_URLS:
        live_urls = pd.read_csv(URL_FILE)
    else:
        with CareerVietCrawler(**LIVE_REQUEST) as crawler:
            live_urls = crawler.collect_urls_from_sitemaps(save_path=str(URL_FILE))
    if "url" not in live_urls.columns:
        raise ValueError("File URL hiện tại thiếu cột url")
    print(f"Hiện tại: {len(live_urls):,} URL")
    display(live_urls.head())
else:
    print("Bỏ qua nguồn hiện tại theo MODE.")


Hiện tại: 590 URL


,url
0,https://careerviet.vn/vi/tim-viec-lam/software...
1,https://careerviet.vn/vi/tim-viec-lam/tt-cntt-...
2,https://careerviet.vn/vi/tim-viec-lam/business...
3,https://careerviet.vn/vi/tim-viec-lam/cloud-de...
4,https://careerviet.vn/vi/tim-viec-lam/chuyen-v...


## 4. Gom snapshot lịch sử

Cell tự bỏ qua nếu chọn `live`. Wayback chỉ có những trang từng được lưu, nên số tin mỗi năm có thể ít hoặc bằng 0. Bộ lọc từ khóa có thể bỏ sót tiêu đề chung chung.

Nếu timeout hoặc 403/429, đọc lỗi và chạy lại cell sau để tiếp tục từ trang CDX đã lưu. Khi lỗi, CSV URL có thể chỉ chứa kết quả một phần. Việc truy cập Wayback và parser HTML cũ chưa được xác minh trực tiếp do timeout trong môi trường phát triển.


In [11]:
if RUN_HISTORY:
    with CareerVietCrawler(**HISTORY_REQUEST) as crawler:
        history_urls = crawler.collect_historical_urls(
            start_year=HISTORY_START_YEAR,
            end_year=HISTORY_END_YEAR,
            save_path=str(HISTORY_URL_FILE),
            resume=True,
        )
    print(f"Lịch sử: {len(history_urls):,} snapshot đã gom")
    display(history_urls.head())
else:
    print("Bỏ qua nguồn lịch sử theo MODE.")


RateLimitedError: HTTP 429 tại https://web.archive.org/cdx/search/cdx?url=careerbuilder.vn%2Fvi%2Ftim-viec-lam%2F%2A&from=2022&to=2022&output=json&fl=timestamp%2Coriginal&filter=statuscode%3A200&filter=mimetype%3Atext%2Fhtml&collapse=urlkey&limit=1000&showResumeKey=true. Đã dừng để giữ checkpoint.

## 5. Crawl chi tiết hiện tại

Chạy lại cell crawl của từng nguồn để xử lý lượt tiếp theo. CSV ghi ngay sau mỗi tin thành công; state lưu định kỳ và khi kết thúc. Có thể Interrupt kernel để dừng. Khi 403/429, crawler dừng và giữ URL chưa xử lý.


In [ ]:
if RUN_LIVE:
    if not URL_FILE.exists():
        raise FileNotFoundError("Chạy cell gom URL hiện tại trước")
    with CareerVietCrawler(**LIVE_REQUEST) as crawler:
        live_jobs = crawler.crawl_jobs_from_url_file(
            url_file_path=str(URL_FILE), save_path=str(JOB_FILE),
            limit=CRAWL_LIMIT, save_every=SAVE_EVERY, resume=RESUME_JOBS,
        )
    print(f"Hiện tại: đã lưu {len(live_jobs):,} tin")
    display(live_jobs.tail())
else:
    print("Bỏ qua nguồn hiện tại theo MODE.")


## 6. Crawl chi tiết lịch sử

Tải HTML snapshot đã chọn. Nếu snapshot chuyển hướng hoặc không đọc được, crawler giữ lại để thử sau; không tự lấy trang live thay thế. Checkpoint nhận diện bằng URL snapshot, cho phép cùng một tin xuất hiện ở các năm khác nhau.


In [ ]:
if RUN_HISTORY:
    if not HISTORY_URL_FILE.exists():
        raise FileNotFoundError("Chạy cell gom snapshot lịch sử trước")
    with CareerVietCrawler(**HISTORY_REQUEST) as crawler:
        history_jobs = crawler.crawl_historical_jobs(
            url_file_path=str(HISTORY_URL_FILE), save_path=str(HISTORY_JOB_FILE),
            limit=CRAWL_LIMIT, save_every=SAVE_EVERY, resume=RESUME_JOBS,
        )
    print(f"Lịch sử: đã lưu {len(history_jobs):,} bản ghi snapshot")
    display(history_jobs.tail())
else:
    print("Bỏ qua nguồn lịch sử theo MODE.")


## 7. Đọc kết quả đã lưu

Cell này không gửi request. Lịch sử có thêm `archive_timestamp` và `archive_url`. Ngày đăng thiếu được để trống, không thay bằng ngày chụp hay ngày cập nhật sitemap. Khi phân tích số job theo năm, dùng `posted_date` và khử trùng theo URL/ID tin; số snapshot không phải số job độc bản.


In [ ]:
for label, enabled, path in (
    ("Hiện tại", RUN_LIVE, JOB_FILE),
    ("Lịch sử", RUN_HISTORY, HISTORY_JOB_FILE),
):
    if not enabled:
        continue
    if not path.exists():
        print(f"{label}: chưa có file kết quả")
        continue
    saved = pd.read_csv(path, keep_default_na=False)
    print(f"{label}: {len(saved):,} bản ghi — {path}")
    display(saved.head(10))
